# Text-as-Data — Coursework 1
## Diagnosing ML Email-Classification Pipelines

**Student:** Nitesh Ranjan Singh  
**Module:** Text-as-Data (MSc Data Science, University of Glasgow)  

---

**Email categories (5 classes):**

| Label | Description |
|---|---|
| Client & Partner Communications | Customers/partners re: relationships, projects, support |
| HR & Internal Admin | Hiring, policies, benefits, internal announcements |
| Operations & Maintenance | Daily ops, issue resolution, maintenance updates |
| Product & Engineering | Building, improving, maintaining products/systems |
| Sales & Marketing | Sales activities, marketing campaigns, lead engagement |

In [ ]:
# Common imports
import numpy as np
import pandas as pd
import torch
from collections import Counter
from itertools import permutations
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Use GPU if available (Colab: Runtime > Change runtime type > T4 GPU)
DEVICE = 0 if torch.cuda.is_available() else -1
BATCH_SIZE = 64 if torch.cuda.is_available() else 16
print(f"Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
def evaluate_model(model_name, dataset_name):
    """Evaluate a model on all splits. Returns results, dataset, model, tokenizer, clf,
    and cached predictions dict {split: list_of_preds} to avoid redundant inference."""
    ds = load_dataset(dataset_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    clf = pipeline("text-classification", model=model, tokenizer=tokenizer,
                   device=DEVICE, batch_size=BATCH_SIZE)

    id2label = model.config.id2label
    print(f"Model id2label: {id2label}")
    labels_are_strings = isinstance(ds['train'][0]['label'], str)
    print(f"Label type: {'string' if labels_are_strings else 'int'}\n")

    results = {}
    all_preds = {}  # cache predictions
    for split_name in ["train", "validation", "test"]:
        if split_name not in ds:
            print(f"  {split_name}: NOT FOUND")
            continue
        texts = list(ds[split_name]["text"])
        true_labels = list(ds[split_name]["label"])

        preds = clf(texts, truncation=True, max_length=512)
        all_preds[split_name] = preds
        pred_labels = [p["label"] for p in preds]

        if labels_are_strings:
            acc = accuracy_score(true_labels, pred_labels)
            all_lbl = sorted(set(true_labels + pred_labels))
            lmap = {l: i for i, l in enumerate(all_lbl)}
            f1 = f1_score([lmap[l] for l in true_labels],
                          [lmap[l] for l in pred_labels], average="macro")
        else:
            label2id = model.config.label2id
            pred_int = [label2id[p] for p in pred_labels]
            acc = accuracy_score(true_labels, pred_int)
            f1 = f1_score(true_labels, pred_int, average="macro")

        results[split_name] = {"accuracy": round(acc, 3), "macro_f1": round(f1, 3)}
        print(f"  {split_name}: Accuracy={acc:.3f}, Macro-F1={f1:.3f}")

    return results, ds, model, tokenizer, clf, all_preds

---
## Q1 — OrbitalForge [2 marks]

**Task:** Identify which tokenizer was used to tokenize the dataset. The dataset is already tokenized (no raw text). Match token IDs against candidate tokenizers.

**Candidates:** `roberta-base`, `microsoft/deberta-base`, `bert-base-uncased`, `distilbert-base-uncased`, `bert-base-cased`, `allenai/scibert_scivocab_uncased`

In [ ]:
# Load the tokenized dataset (no model inference needed — fast)
ds_q1 = load_dataset("TextAsData/Q1-OrbitalForge-dataset")
print("Columns:", ds_q1['train'].column_names)

sample_ids = ds_q1['train'][0]['input_ids']
print(f"Sample input_ids (first 20): {sample_ids[:20]}")
print(f"Sequence length: {len(sample_ids)}")

# First/last non-zero token across examples
for i in range(3):
    ids = ds_q1['train'][i]['input_ids']
    nonzero = [x for x in ids if x != 0]
    print(f"  Ex {i}: first={ids[0]}, last_nonzero={nonzero[-1]}")

all_ids = [tid for ex in ds_q1['train'] for tid in ex['input_ids'] if tid != 0]
print(f"\nMax token ID: {max(all_ids)}")

In [ ]:
# Decode sample using each candidate (tokenizer-only, no model inference — fast)
candidates = [
    "roberta-base", "microsoft/deberta-base", "bert-base-uncased",
    "distilbert-base-uncased", "bert-base-cased", "allenai/scibert_scivocab_uncased"
]
sample_ids = ds_q1['train'][0]['input_ids']

for name in candidates:
    tok = AutoTokenizer.from_pretrained(name)
    decoded = tok.decode(sample_ids, skip_special_tokens=True)
    print(f"\n{name} (vocab={tok.vocab_size}, CLS={tok.cls_token_id}, SEP={tok.sep_token_id}):")
    print(f"  {decoded[:200]}")

### Q1 — Answer

**The tokenizer used is `bert-base-cased`.**

**Evidence:**
1. **Special token IDs match:** Every example starts with token ID 101 (CLS) and ends with 102 (SEP). These are the CLS/SEP IDs for `bert-base-cased` (and `bert-base-uncased`/`distilbert-base-uncased`). RoBERTa uses 0/2, DeBERTa uses 1/2, and SciBERT uses 102/103 — ruling them out.
2. **Decoded text is coherent:** Only `bert-base-cased` produces readable English when decoding the token IDs (e.g., "SUBJECT: Catch-up on Friday?..."). The other BERT-based tokenizers with matching special token IDs (`bert-base-uncased`, `distilbert-base-uncased`) produce garbled `[unused]` tokens because their vocabularies map the same IDs to different tokens.
3. **Vocab size compatibility:** The max token ID in the dataset (28,122) fits within `bert-base-cased`'s vocab of 28,996, and is notably close to its maximum — consistent with using the full cased BERT vocabulary.

---
## Q2 — NeuroBloom [3 marks]

**Task:** A "brand-new BERT from scratch" with a "bespoke tokenizer" was claimed. Find what's wrong with the tokenizer.

In [ ]:
# Load model and tokenizer (no inference needed for this question — fast)
tokenizer_q2 = AutoTokenizer.from_pretrained("TextAsData/Q2-NeuroBloom-model")
model_q2 = AutoModelForSequenceClassification.from_pretrained("TextAsData/Q2-NeuroBloom-model")

print(f"Tokenizer type: {type(tokenizer_q2).__name__}")
print(f"Vocab size (tokenizer): {tokenizer_q2.vocab_size}")
print(f"Model embedding shape: {model_q2.get_input_embeddings().weight.shape}")
print(f"Special tokens: CLS={tokenizer_q2.cls_token}, SEP={tokenizer_q2.sep_token}, UNK={tokenizer_q2.unk_token}")

# Vocab analysis
vocab = tokenizer_q2.get_vocab()
non_special = [t for t in vocab if not t.startswith('[')]
length_dist = Counter(len(t) for t in non_special)
print(f"\nToken length distribution: {dict(sorted(length_dist.items()))}")
print(f"Tokens of length >= 3: {[t for t in non_special if len(t) >= 3]}")

# Common word check
for w in ['the', 'and', 'email', 'meeting', 'project', 'team']:
    print(f"  '{w}' in vocab: {w in vocab}")

In [ ]:
# UNK rate test (tokenizer-only, no model inference — fast)
ds_q2 = load_dataset("TextAsData/Q2-NeuroBloom-dataset")

for i in range(3):
    tokens = tokenizer_q2.tokenize(ds_q2['train'][i]['text'])
    unk_count = tokens.count('[UNK]')
    print(f"Ex {i}: {len(tokens)} tokens, {unk_count} UNK ({unk_count/len(tokens)*100:.1f}%)")
    print(f"  First 20 tokens: {tokens[:20]}\n")

# Overall UNK rate
total_tokens = total_unks = 0
for i in range(200):
    tokens = tokenizer_q2.tokenize(ds_q2['train'][i]['text'])
    total_tokens += len(tokens)
    total_unks += tokens.count('[UNK]')
print(f"Overall UNK rate (200 examples): {total_unks}/{total_tokens} = {total_unks/total_tokens*100:.1f}%")

### Q2 — Answer

**The "bespoke" tokenizer is fundamentally broken — it has a tiny vocabulary of only 1,145 tokens consisting entirely of single and double-character entries.**

**Key findings:**
1. **Extremely small vocab (1,145 tokens):** A standard BERT tokenizer has ~30,000 tokens. This tokenizer has only 1,145 — far too small for any real NLP task.
2. **Only 1-2 character tokens:** The entire vocabulary consists of single characters (779 tokens) and two-character combinations (361 tokens), plus 5 special tokens. There are zero tokens of length 3 or more. This means the tokenizer was never properly trained using BPE/WordPiece on any text corpus.
3. **78% UNK rate:** When tokenizing email texts, approximately 78% of all tokens are mapped to `[UNK]`. Common English words like "the", "and", "email", "meeting", and "project" are not in the vocabulary at all.
4. **Unusable for classification:** With such a high UNK rate, the model receives almost no meaningful input — the vast majority of tokens are `[UNK]`, destroying all semantic information needed for email classification.

**Root cause:** The tokenizer was not trained on any substantial text corpus. It appears to be a character-level vocabulary (all single letters and some bigrams) rather than a properly trained WordPiece tokenizer. Despite the model embedding matrix matching the tokenizer vocab size (1,145), the tokenizer itself cannot represent English text, making the entire pipeline unusable.

---
## Q3 — HydroVine [4 marks]

**Task:** The model isn't performing well. Diagnose why.

In [ ]:
# Evaluate (small dataset: 1400+300+300 = 2000 examples — relatively fast)
results_q3, ds_q3, model_q3, tokenizer_q3, clf_q3, preds_q3 = evaluate_model(
    "TextAsData/Q3-HydroVine-model", "TextAsData/Q3-HydroVine-dataset"
)

In [ ]:
# Analysis using CACHED predictions (no re-inference)
for split in ['train', 'validation', 'test']:
    dist = Counter(ds_q3[split]['label'])
    print(f"{split}: {dict(sorted(dist.items()))}")

# Per-class test report from cached preds
true_labels = list(ds_q3['test']['label'])
pred_labels = [p['label'] for p in preds_q3['test']]
print("\nTest set classification report:")
print(classification_report(true_labels, pred_labels))

# Model details
print(f"Total params: {sum(p.numel() for p in model_q3.parameters()):,}")
print(f"Train size: {len(ds_q3['train'])}")

### Q3 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 1.000 | 1.000 |
| Validation | 0.470 | 0.468 |
| Test | 0.497 | 0.493 |

**Diagnosis: Severe overfitting.**

The model achieves perfect performance on the training set (100% accuracy) but only ~50% on validation and test — barely better than random for a 5-class problem. This is a textbook case of overfitting:

1. **Very small training set (1,400 examples)** combined with a large BERT model (~109M parameters). The model has far more parameters than training examples, making memorization trivial.
2. **Perfect train accuracy (1.000)** confirms the model has memorized the training data rather than learning generalizable patterns.
3. **Val/test ~0.50** shows the model has learned nothing transferable — its predictions on unseen data are essentially random.
4. **No tokenizer or label mapping issues** — the tokenizer is standard `bert-base-uncased` with matching vocab, and labels are consistent between model and dataset.

**The fix would be:** more training data, regularization (dropout, weight decay), early stopping based on validation loss, data augmentation, or using a smaller model.

---
## Q4 — LumenGrid [4 marks]

**Task:** Client says deployed performance doesn't match reported test performance. Evaluate and decide if something is wrong.

In [ ]:
# Evaluate (6300+1350+1350 = 9000 examples)
results_q4, ds_q4, model_q4, tokenizer_q4, clf_q4, preds_q4 = evaluate_model(
    "TextAsData/Q4-LumenGrid-model", "TextAsData/Q4-LumenGrid-dataset"
)

In [ ]:
# Data leakage check (no inference — fast)
train_texts = set(ds_q4['train']['text'])
val_texts = set(ds_q4['validation']['text'])
test_texts = set(ds_q4['test']['text'])
print(f"Train: {len(ds_q4['train'])} examples, {len(train_texts)} unique")
print(f"Train ∩ Test: {len(train_texts & test_texts)}")
print(f"Train ∩ Val: {len(train_texts & val_texts)}")

# Confidence analysis from CACHED predictions (no re-inference)
for split in ['train', 'validation', 'test']:
    confs = [p['score'] for p in preds_q4[split]]
    true_labels = list(ds_q4[split]['label'])
    correct = sum(1 for p, l in zip(preds_q4[split], true_labels) if p['label'] == l)
    print(f"{split}: mean_conf={np.mean(confs):.3f}, errors={len(true_labels)-correct}")

### Q4 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 0.822 | 0.826 |
| Validation | 0.818 | 0.822 |
| Test | 0.982 | 0.983 |

**Diagnosis: Data leakage — the test set was likely included in the training data used to build the model.**

**Evidence:**
1. **Inverted performance pattern:** Test accuracy (0.982) is dramatically higher than both train (0.822) and validation (0.818). In a properly trained model, train performance should be highest (or at least equal). Having test >> train is a hallmark of data leakage.
2. **Confidence gap:** The model's mean prediction confidence on the test set (0.987) is significantly higher than on train (0.960), with only 24 errors on test vs 1,122 on train. This indicates the model has effectively "seen" the test examples during training.
3. **Minimal text overlap:** Only 1 exact duplicate exists between train and test, so the leakage is not from identical copies in the provided dataset splits. Rather, the model was trained on data that included the test set — either the actual training data used for model fitting contained the test examples, or the test set was used for model selection/tuning.
4. **Deployed performance would match train/val (~82%)**, not the inflated test result (98.2%), explaining the client's complaint about deployment performance not matching reported metrics.

**The reported test performance (98.2%) is not trustworthy.** The real generalisation performance is approximately 82% (as shown by train and validation).

---
## Q5 — AeroSynth [6 marks]

**Task:** Client encountered "unusual performance results." Find what's unusual and what causes it.

In [ ]:
# Evaluate (2100+450+450 = 3000 examples)
results_q5, ds_q5, model_q5, tokenizer_q5, clf_q5, preds_q5 = evaluate_model(
    "TextAsData/Q5-AeroSynth-model", "TextAsData/Q5-AeroSynth-dataset"
)

In [ ]:
# Per-class analysis from CACHED predictions (no re-inference)
for split in ['train', 'validation', 'test']:
    true_labels = list(ds_q5[split]['label'])
    pred_labels = [p['label'] for p in preds_q5[split]]
    print(f"\n{'='*60}\n{split.upper()}:")
    print(classification_report(true_labels, pred_labels))

In [ ]:
# Check if label names are embedded in the text (no inference — fast)
print("Sample texts from perfect-score classes:\n")
for label in ['HR & Internal Admin', 'Product & Engineering', 'Client & Partner Communications']:
    for i in range(len(ds_q5['test'])):
        if ds_q5['test'][i]['label'] == label:
            print(f"{label}:")
            print(f"  {ds_q5['test'][i]['text'][:150]}\n")
            break

# Count how many texts start with the label name
print("Label name prefix check:")
for label_name in sorted(set(ds_q5['train']['label'])):
    examples = [ds_q5['train'][i]['text'] for i in range(len(ds_q5['train']))
                if ds_q5['train'][i]['label'] == label_name]
    starts = sum(1 for t in examples if t.startswith(label_name))
    print(f"  {label_name}: {starts}/{len(examples)} ({starts/len(examples)*100:.0f}%) start with label name")

### Q5 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 0.894 | 0.909 |
| Validation | 0.838 | 0.853 |
| Test | 0.818 | 0.836 |

**What's unusual:**
1. **Two classes have perfect F1 (1.00) across ALL splits**, while the other three have much lower performance:
   - HR & Internal Admin: **1.00** precision/recall/F1 on train, val, and test
   - Product & Engineering: **1.00** precision/recall/F1 on train, val, and test
   - Client & Partner Communications: ~0.57 F1 (test)
   - Operations & Maintenance: ~0.90 F1 (test)
   - Sales & Marketing: ~0.70 F1 (test)

2. **Macro-F1 > Accuracy** — This is unusual. Normally accuracy >= macro-F1. Here, the two perfect classes pull up the macro-F1 average, while the model's errors on the remaining classes reduce overall accuracy.

**What causes it: Label leakage in the text data.**

The texts for "HR & Internal Admin" and "Product & Engineering" have the **category name literally prepended to the email text** (e.g., `"HR & Internal Admin SUBJECT:Friday Catch-up..."` and `"Product & Engineering SUBJECT:Demo Prep..."`). The model trivially learns to match these prefixes, achieving perfect classification. The other three classes do not have this prefix, so the model must actually understand the email content — and achieves much lower performance on those.

This is a data quality issue where label information was accidentally (or carelessly) embedded directly into the input features for 2 out of 5 classes.

---
## Q6 — CryoNest [4 marks]

**Task:** Client believes there's a dataset problem causing evaluation issues. Diagnose it.

In [ ]:
# First check for the dataset problem BEFORE running expensive inference
ds_q6 = load_dataset("TextAsData/Q6-CryoNest-dataset")

train_set = set(ds_q6['train']['text'])
val_set = set(ds_q6['validation']['text'])
test_set = set(ds_q6['test']['text'])

print(f"Train: {len(ds_q6['train'])} examples, {len(train_set)} unique")
print(f"Val:   {len(ds_q6['validation'])} examples, {len(val_set)} unique")
print(f"Test:  {len(ds_q6['test'])} examples, {len(test_set)} unique")
print(f"\nTrain ∩ Val: {len(train_set & val_set)}")
print(f"Train ∩ Test: {len(train_set & test_set)}")
print(f"Val ∩ Test: {len(val_set & test_set)}")
print(f"\nAll three splits identical: {train_set == val_set == test_set}")

# Class distributions
for split in ['train', 'validation', 'test']:
    dist = Counter(ds_q6[split]['label'])
    print(f"{split}: {dict(sorted(dist.items()))}")

In [ ]:
# Since all splits are identical, we only need to run inference ONCE on train
# (instead of 3x on identical data)
tokenizer_q6 = AutoTokenizer.from_pretrained("TextAsData/Q6-CryoNest-model")
model_q6 = AutoModelForSequenceClassification.from_pretrained("TextAsData/Q6-CryoNest-model")
clf_q6 = pipeline("text-classification", model=model_q6, tokenizer=tokenizer_q6,
                  device=DEVICE, batch_size=BATCH_SIZE)

texts = list(ds_q6['train']['text'])
true_labels = list(ds_q6['train']['label'])
preds = clf_q6(texts, truncation=True, max_length=512)
pred_labels = [p['label'] for p in preds]

acc = accuracy_score(true_labels, pred_labels)
all_lbl = sorted(set(true_labels + pred_labels))
lmap = {l: i for i, l in enumerate(all_lbl)}
f1 = f1_score([lmap[l] for l in true_labels], [lmap[l] for l in pred_labels], average='macro')

# Same result for all 3 splits since data is identical
print(f"All splits (identical data): Accuracy={acc:.3f}, Macro-F1={f1:.3f}")

### Q6 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 0.855 | 0.861 |
| Validation | 0.855 | 0.861 |
| Test | 0.855 | 0.861 |

**Diagnosis: All three splits (train, validation, test) contain the exact same 2,800 examples.**

**Evidence:**
1. **Identical metrics:** All three splits produce exactly the same accuracy (0.855) and macro-F1 (0.861), which would be astronomically unlikely if they contained different data.
2. **100% overlap:** The train, validation, and test sets all contain 2,800 examples, and the intersection of any two splits is 2,800 — they are identical copies.
3. **Identical class distributions:** Each split has exactly the same class counts (659 Client & Partner, 479 HR, 433 Operations, 599 Product, 630 Sales).

**Impact:** There is no way to evaluate model generalisation because there is no held-out data. The reported test performance is simply the training performance. Any model trained on this data would appear to perform well on "test" since it's the same data.

**The fix:** Create proper train/validation/test splits with no overlap between them.

---
## Q7 — NeonPixel [5 marks]

**Task:** Is the model ready to deploy? If not, what's the issue?

In [ ]:
# Evaluate (2924+900+900 = 4724 examples)
results_q7, ds_q7, model_q7, tokenizer_q7, clf_q7, preds_q7 = evaluate_model(
    "TextAsData/Q7-NeonPixel-model", "TextAsData/Q7-NeonPixel-dataset"
)

In [ ]:
# Class distributions and per-class report from CACHED preds (no re-inference)
for split in ['train', 'validation', 'test']:
    dist = Counter(ds_q7[split]['label'])
    total = sum(dist.values())
    print(f"\n{split} ({total} examples):")
    for label in sorted(dist.keys()):
        print(f"  {label}: {dist[label]} ({dist[label]/total*100:.1f}%)")

# Test report from cached preds
true_labels = list(ds_q7['test']['label'])
pred_labels = [p['label'] for p in preds_q7['test']]
print("\nTest classification report:")
print(classification_report(true_labels, pred_labels))
print(f"Prediction distribution: {Counter(pred_labels)}")

### Q7 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 0.936 | 0.935 |
| Validation | 0.586 | 0.416 |
| Test | 0.591 | 0.420 |

**The model is NOT ready to deploy. The training data is missing 2 out of 5 classes.**

**Key findings:**
1. **Training set has only 3 classes:** The train split contains only "Client & Partner Communications" (957), "Product & Engineering" (1,050), and "Sales & Marketing" (917). The classes "HR & Internal Admin" and "Operations & Maintenance" are completely absent from the training data.
2. **Validation and test have all 5 classes:** Both val and test include all 5 email categories, including the two missing from training.
3. **Zero performance on missing classes:** The model achieves 0.00 precision, recall, and F1 for "HR & Internal Admin" and "Operations & Maintenance" — it never predicts these classes because it was never trained on them.
4. **Inflated train accuracy (93.6%)** is misleading because the model is only evaluated on the 3 classes it knows.
5. **Large accuracy-F1 gap:** Test accuracy (0.591) vs macro-F1 (0.420) — the gap exists because the 3 known classes contribute to accuracy, but the 2 unknown classes drag macro-F1 down.
6. **The model only predicts 3 labels**, funnelling all HR/Operations examples into incorrect categories.

**The fix:** Retrain the model with training data that includes all 5 email categories.

---
## Q8 — SolaraMesh [5 marks]

**Task:** Client was told 94% accuracy and ready to deploy. Was this reported in good faith?

In [ ]:
# Evaluate (small dataset: 1033+221+222 = 1476 examples — fast)
results_q8, ds_q8, model_q8, tokenizer_q8, clf_q8, preds_q8 = evaluate_model(
    "TextAsData/Q8-SolaraMesh-model", "TextAsData/Q8-SolaraMesh-dataset"
)

In [ ]:
# Class distribution + per-class report from CACHED preds (no re-inference)
for split in ['train', 'validation', 'test']:
    dist = Counter(ds_q8[split]['label'])
    total = sum(dist.values())
    print(f"\n{split} ({total} examples):")
    for label in sorted(dist.keys()):
        print(f"  {label}: {dist[label]} ({dist[label]/total*100:.1f}%)")

# Train report from cached preds
true_labels = list(ds_q8['train']['label'])
pred_labels = [p['label'] for p in preds_q8['train']]
print("\nTrain classification report:")
print(classification_report(true_labels, pred_labels))

print("Accuracy vs Macro-F1 gap:")
for split, metrics in results_q8.items():
    print(f"  {split}: Acc={metrics['accuracy']}, F1={metrics['macro_f1']}, Gap={metrics['accuracy']-metrics['macro_f1']:.3f}")

### Q8 — Answer

**Metrics:**

| Split | Accuracy | Macro-F1 |
|---|---|---|
| Train | 0.941 | 0.292 |
| Validation | 0.923 | 0.274 |
| Test | 0.923 | 0.224 |

**The 94% accuracy claim was NOT reported in good faith. It is deeply misleading.**

**What's happening:**
1. **Extreme class imbalance:** The dataset is overwhelmingly dominated by "Sales & Marketing", which makes up ~92% of all examples across every split. The remaining 4 classes each represent only 1-2% of the data.
2. **Accuracy is misleading:** The ~94% accuracy is achieved simply because the model almost always predicts "Sales & Marketing" (the majority class). A trivial baseline that always predicts "Sales & Marketing" would achieve ~92% accuracy with zero learning.
3. **Macro-F1 reveals the truth:** The macro-F1 scores (0.292 train, 0.274 validation, 0.224 test) show the model's true quality — it performs terribly. It gets 0.00 F1 for 3 out of 5 classes (Client & Partner, HR, Operations) and only ~0.14-0.47 for Product & Engineering.
4. **Massive accuracy-F1 gap:** The gap between accuracy (0.941) and macro-F1 (0.292) is 0.649 — this enormous discrepancy is the hallmark of class imbalance making accuracy a deceptive metric.

**Conclusion:** Reporting 94% accuracy without disclosing the extreme class imbalance (92% majority class) or reporting macro-F1 is misleading. The model essentially acts as a majority-class classifier and would fail catastrophically on any class other than Sales & Marketing. This was not reported in good faith because any competent evaluation would include macro-F1 or per-class metrics alongside accuracy for such an imbalanced dataset.

---
## Q9 — Time Taken [0 marks]

**Estimated time spent:** 5 hours